# MNIST score-based generation in a 64D PCA space

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.5, Appendix F, Table 11.

Variance-preserving score-based diffusion in whitened 64D PCA space. Gaeta–Lie regularizes the reverse-SDE drift with generators from a 434-template library; the held-out adequacy diagnostic selects m. Compared against score-only continuation with equal additional updates.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Configuration, imports, reproducibility, and shared utilities

In [ ]:
# ============================================================================
# Cell 1 — Configuration, imports, reproducibility, and shared utilities
# ============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)
jax_config.update("jax_default_matmul_precision", "high")

import os
import gc
import gzip
import json
import math
import time
import struct
import urllib.request
from pathlib import Path
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
if not any(d.platform == "gpu" for d in jax.devices()):
    print("WARNING: no GPU detected. Select Runtime > Change runtime type > GPU.")


CONFIG = {
    # ------------------------------------------------------------------
    # Reproducibility and run mode
    # ------------------------------------------------------------------
    "SEED": 42,
    "FAST_MODE": False,
    "RUN_MULTI_SEED_CONFIRMATION": False,
    "CONFIRMATION_SEEDS": (101, 202, 303),

    # ------------------------------------------------------------------
    # Data and fixed latent representation
    # ------------------------------------------------------------------
    "DATA_DIR": "./gaeta_lie_mnist_data",
    "TRAIN_COUNT": 50_000,
    "VAL_COUNT": 10_000,
    "LATENT_DIM": 64,
    "PCA_EPS": 1e-5,
    "PCA_BATCH_SIZE": 5_000,
    "DEQUANTIZATION": True,

    # ------------------------------------------------------------------
    # VP forward SDE: dZ_t = -0.5 beta(t) Z_t dt + sqrt(beta(t)) dW_t
    # ------------------------------------------------------------------
    "BETA_MIN": 0.1,
    "BETA_MAX": 20.0,
    "TIME_EPS": 0.02,

    # ------------------------------------------------------------------
    # Score network and Stage-I DSM training
    # ------------------------------------------------------------------
    "TIME_EMBED_DIM": 32,
    "SCORE_HIDDEN_DIM": 256,
    "SCORE_RES_BLOCKS": 4,
    "BASELINE_UPDATES": 15_000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-4,
    "BASELINE_EMA_DECAY": 0.999,
    "BASELINE_GRAD_CLIP": 1.0,
    "BASELINE_PRINT_EVERY": 1_000,
    "BASELINE_VAL_BATCHES": 8,

    # ------------------------------------------------------------------
    # Stage-II affine projectable library and discovery
    # n=64, and only m in {n-20,n-10,n} = {44,54,64} is considered.
    # ------------------------------------------------------------------
    "GENERATOR_COUNTS": (44, 54, 64),
    "BASIS_MU_LIST": (-1.0, -0.5, 0.5, 1.0),
    "HADAMARD_DIRECTIONS": 8,
    "BASIS_EXP_CLIP": 8.0,

    "DISCOVERY_TRAIN_POINTS": 256,
    "DISCOVERY_VAL_POINTS": 256,
    "DISCOVERY_PROBES": 2,
    "DISCOVERY_SPECTRAL_RIDGE": 1e-3,
    "DISCOVERY_ADAM_STEPS": 150,
    "DISCOVERY_ADAM_LR": 3e-3,
    "DISCOVERY_CLOSURE_POINTS": 8,
    "DISCOVERY_CLOSURE_PAIRS": 96,
    "DISCOVERY_LAMBDA_DET": 1.0,
    "DISCOVERY_LAMBDA_CLOSE": 0.02,
    "DISCOVERY_LAMBDA_IND": 0.25,
    "DISCOVERY_LAMBDA_NORM": 0.01,
    "DISCOVERY_MIN_NORM": 0.50,
    "DISCOVERY_CLOSURE_RIDGE": 1e-4,
    "DISCOVERY_PRINT_EVERY": 50,

    # Data-only adequacy gate.
    "ADEQUACY_RANDOM_SUBSPACES": 24,
    "ADEQUACY_RANDOM_QUANTILE": 0.05,
    "ADEQUACY_MAX_VAL_TRAIN_RATIO": 2.0,
    "ADEQUACY_BOOTSTRAPS": 5,
    "ADEQUACY_MAX_BOOTSTRAP_CV": 0.35,
    "ADEQUACY_EXACT_POINTS": 8,

    # ------------------------------------------------------------------
    # Stage-III matched continuation and Gaeta-Lie refinement
    # ------------------------------------------------------------------
    "REFINE_UPDATES": 1_000,
    "REFINE_BATCH_SIZE": 512,
    "REFINE_LR": 1e-4,
    "REFINE_EMA_DECAY": 0.999,
    "REFINE_GRAD_CLIP": 1.0,
    "REFINE_SYM_POINTS": 32,
    "REFINE_HUTCHINSON_PROBES": 2,
    "REFINE_WARMUP_FRACTION": 0.20,
    "REFINE_PRINT_EVERY": 250,

    # Lambda is selected once on m=n-10=54 and then held fixed for all m.
    "PILOT_M": 54,
    "PILOT_LAMBDAS": (0.01, 0.03, 0.10),
    "PILOT_UPDATES": 200,
    "PILOT_VALIDATION_WEIGHT_DET": 0.10,

    # ------------------------------------------------------------------
    # Reverse-SDE sampling
    # ------------------------------------------------------------------
    "SAMPLER_STEPS": 100,
    "METRIC_SAMPLES": 5_000,
    "VALIDATION_SELECTION_SAMPLES": 2_000,
    "DISPLAY_SAMPLES": 64,

    # ------------------------------------------------------------------
    # Frozen MNIST evaluator and generative metrics
    # ------------------------------------------------------------------
    "CLASSIFIER_HIDDEN_1": 256,
    "CLASSIFIER_FEATURE_DIM": 128,
    "CLASSIFIER_EPOCHS": 5,
    "CLASSIFIER_BATCH_SIZE": 512,
    "CLASSIFIER_LR": 1e-3,
    "CLASSIFIER_PRINT_EVERY": 1,
    "SWD_PROJECTIONS": 128,
    "NN_GENERATED_SUBSET": 500,
    "NN_TRAIN_SUBSET": 5_000,

    # ------------------------------------------------------------------
    # Output
    # ------------------------------------------------------------------
    "RESULTS_JSON": "./gaeta_lie_mnist64_results.json",
}


if CONFIG["FAST_MODE"]:
    # This mode is only for checking that all cells execute end to end.
    # Do not use FAST_MODE numbers for reported results.
    CONFIG.update({
        "TRAIN_COUNT": 12_000,
        "VAL_COUNT": 2_000,
        "BASELINE_UPDATES": 600,
        "BASELINE_BATCH_SIZE": 256,
        "BASELINE_PRINT_EVERY": 100,
        "BASELINE_VAL_BATCHES": 2,
        "DISCOVERY_TRAIN_POINTS": 64,
        "DISCOVERY_VAL_POINTS": 64,
        "DISCOVERY_PROBES": 1,
        "DISCOVERY_ADAM_STEPS": 20,
        "DISCOVERY_CLOSURE_POINTS": 4,
        "DISCOVERY_CLOSURE_PAIRS": 24,
        "ADEQUACY_RANDOM_SUBSPACES": 6,
        "ADEQUACY_BOOTSTRAPS": 3,
        "ADEQUACY_EXACT_POINTS": 2,
        "PILOT_UPDATES": 20,
        "REFINE_UPDATES": 80,
        "REFINE_BATCH_SIZE": 256,
        "REFINE_SYM_POINTS": 8,
        "REFINE_HUTCHINSON_PROBES": 1,
        "REFINE_PRINT_EVERY": 20,
        "SAMPLER_STEPS": 20,
        "METRIC_SAMPLES": 500,
        "VALIDATION_SELECTION_SAMPLES": 300,
        "DISPLAY_SAMPLES": 25,
        "CLASSIFIER_EPOCHS": 1,
        "CLASSIFIER_BATCH_SIZE": 256,
        "SWD_PROJECTIONS": 32,
        "NN_GENERATED_SUBSET": 100,
        "NN_TRAIN_SUBSET": 1_000,
        "RUN_MULTI_SEED_CONFIRMATION": False,
    })


N = int(CONFIG["LATENT_DIM"])
assert tuple(CONFIG["GENERATOR_COUNTS"]) == (N - 20, N - 10, N)
assert CONFIG["PILOT_M"] == N - 10

GLOBAL_KEY = jax.random.PRNGKey(CONFIG["SEED"])


def next_key():
    """Return a fresh global PRNG subkey for host-controlled loops."""
    global GLOBAL_KEY
    GLOBAL_KEY, subkey = jax.random.split(GLOBAL_KEY)
    return subkey


def tree_copy(tree):
    return jax.tree_util.tree_map(lambda x: jnp.array(x), tree)


def tree_zeros_like(tree):
    return jax.tree_util.tree_map(jnp.zeros_like, tree)


def tree_global_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum(jnp.sum(x * x) for x in leaves) + 1e-12)


def clip_tree_by_global_norm(tree, max_norm):
    norm = tree_global_norm(tree)
    scale = jnp.minimum(1.0, max_norm / (norm + 1e-12))
    return jax.tree_util.tree_map(lambda x: x * scale, tree), norm


def glorot_uniform(key, in_dim, out_dim):
    limit = jnp.sqrt(6.0 / float(in_dim + out_dim))
    return jax.random.uniform(
        key, (in_dim, out_dim), minval=-limit, maxval=limit,
        dtype=jnp.float32,
    )


def adam_init(params):
    return {
        "m": tree_zeros_like(params),
        "v": tree_zeros_like(params),
        "t": jnp.array(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"], grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"], grads,
    )
    b1_correction = 1.0 - beta1 ** t_new
    b2_correction = 1.0 - beta2 ** t_new
    m_hat = jax.tree_util.tree_map(lambda m: m / b1_correction, m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / b2_correction, v_new)
    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params, m_hat, v_hat,
    )
    return params_new, {"m": m_new, "v": v_new, "t": t_new}


def ema_update(ema_params, params, decay):
    return jax.tree_util.tree_map(
        lambda e, p: decay * e + (1.0 - decay) * p,
        ema_params, params,
    )


def rademacher(key, shape):
    return (2 * jax.random.bernoulli(key, 0.5, shape).astype(jnp.float32) - 1.0)


def format_seconds(seconds):
    seconds = float(seconds)
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {sec:.0f}s"
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes)}m"


print("FAST_MODE:", CONFIG["FAST_MODE"])
print("Latent dimension n:", N)
print("Generator sweep m:", CONFIG["GENERATOR_COUNTS"])
print("Multi-seed confirmation enabled:", CONFIG["RUN_MULTI_SEED_CONFIRMATION"])


## Cell 2 — Direct MNIST download, train/validation split, 64D PCA whitening

In [ ]:
# ============================================================================
# Cell 2 — Direct MNIST download, train/validation split, 64D PCA whitening
# ============================================================================

DATA_DIR = Path(CONFIG["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

MNIST_URLS = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}


def download_file(url, path):
    if path.exists() and path.stat().st_size > 0:
        return
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, path)


for name, url in MNIST_URLS.items():
    download_file(url, DATA_DIR / f"{name}.gz")


def read_idx_images(path):
    with gzip.open(path, "rb") as f:
        magic, n_images, n_rows, n_cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected image magic number {magic} in {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n_images, n_rows * n_cols)


def read_idx_labels(path):
    with gzip.open(path, "rb") as f:
        magic, n_labels = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected label magic number {magic} in {path}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n_labels)


mnist_train_images_u8 = read_idx_images(DATA_DIR / "train_images.gz")
mnist_train_labels_np = read_idx_labels(DATA_DIR / "train_labels.gz")
mnist_test_images_u8 = read_idx_images(DATA_DIR / "test_images.gz")
mnist_test_labels_np = read_idx_labels(DATA_DIR / "test_labels.gz")

assert mnist_train_images_u8.shape == (60_000, 784)
assert mnist_test_images_u8.shape == (10_000, 784)

# Work in [-1,1]. The original uint8 arrays stay on CPU.
mnist_train_images_np = mnist_train_images_u8.astype(np.float32) / 127.5 - 1.0
mnist_test_images_np = mnist_test_images_u8.astype(np.float32) / 127.5 - 1.0

split_rng = np.random.default_rng(CONFIG["SEED"])
perm = split_rng.permutation(60_000)
train_count = int(CONFIG["TRAIN_COUNT"])
val_count = int(CONFIG["VAL_COUNT"])
if train_count + val_count > 60_000:
    raise ValueError("TRAIN_COUNT + VAL_COUNT cannot exceed 60,000.")

train_idx = perm[:train_count]
val_idx = perm[train_count:train_count + val_count]

train_x_pixels_np = mnist_train_images_np[train_idx]
train_labels_np = mnist_train_labels_np[train_idx]
val_x_pixels_np = mnist_train_images_np[val_idx]
val_labels_np = mnist_train_labels_np[val_idx]
test_x_pixels_np = mnist_test_images_np
test_labels_np = mnist_test_labels_np

print("Pixel splits:")
print("  train:", train_x_pixels_np.shape)
print("  val  :", val_x_pixels_np.shape)
print("  test :", test_x_pixels_np.shape)


# Fit PCA on training pixels only. The covariance eigendecomposition is 784x784;
# the expensive matrix product is performed by JAX/XLA, preferably on the GPU.
t_pca = time.time()
train_x_device = jnp.asarray(train_x_pixels_np)
PCA_MEAN = jnp.mean(train_x_device, axis=0)
centered_train = train_x_device - PCA_MEAN
PCA_COV = (centered_train.T @ centered_train) / max(1, train_count - 1)
all_evals, all_evecs = jnp.linalg.eigh(PCA_COV)
order = jnp.argsort(all_evals)[::-1]
all_evals = all_evals[order]
all_evecs = all_evecs[:, order]

PCA_EIGVALS = jnp.maximum(all_evals[:N], CONFIG["PCA_EPS"])
PCA_COMPONENTS = all_evecs[:, :N]
PCA_SCALES = jnp.sqrt(PCA_EIGVALS)
PCA_EXPLAINED_RATIO = float(
    jnp.sum(PCA_EIGVALS) / jnp.maximum(jnp.sum(jnp.maximum(all_evals, 0.0)), 1e-12)
)

# Delete the large temporary device matrix before score-model training.
del train_x_device, centered_train, PCA_COV, all_evals, all_evecs
gc.collect()


@jax.jit
def pca_encode_device(x_pixels):
    return ((x_pixels - PCA_MEAN) @ PCA_COMPONENTS) / PCA_SCALES


@jax.jit
def pca_decode_device(z):
    return PCA_MEAN + (z * PCA_SCALES) @ PCA_COMPONENTS.T


def encode_host_array(x_np, batch_size=None):
    batch_size = int(batch_size or CONFIG["PCA_BATCH_SIZE"])
    chunks = []
    for start in range(0, len(x_np), batch_size):
        z = pca_encode_device(jnp.asarray(x_np[start:start + batch_size]))
        chunks.append(np.asarray(z, dtype=np.float32))
    return np.concatenate(chunks, axis=0)


def decode_host_array(z_np, batch_size=None):
    batch_size = int(batch_size or CONFIG["PCA_BATCH_SIZE"])
    chunks = []
    for start in range(0, len(z_np), batch_size):
        x = pca_decode_device(jnp.asarray(z_np[start:start + batch_size]))
        chunks.append(np.asarray(x, dtype=np.float32))
    return np.concatenate(chunks, axis=0)


train_z_np = encode_host_array(train_x_pixels_np)
val_z_np = encode_host_array(val_x_pixels_np)
test_z_np = encode_host_array(test_x_pixels_np)

# Keep compact 64D arrays and labels on device for all model training.
TRAIN_Z = jnp.asarray(train_z_np)
VAL_Z = jnp.asarray(val_z_np)
TEST_Z = jnp.asarray(test_z_np)
TRAIN_LABELS = jnp.asarray(train_labels_np, dtype=jnp.int32)
VAL_LABELS = jnp.asarray(val_labels_np, dtype=jnp.int32)
TEST_LABELS = jnp.asarray(test_labels_np, dtype=jnp.int32)

# Fixed-PCA reconstruction ceiling.
recon_test_np = decode_host_array(test_z_np)
pca_test_mse = float(np.mean((recon_test_np - test_x_pixels_np) ** 2))

print(f"PCA fit/encode time: {format_seconds(time.time() - t_pca)}")
print(f"64D explained variance ratio: {PCA_EXPLAINED_RATIO:.6f}")
print(f"Test reconstruction pixel MSE in [-1,1] space: {pca_test_mse:.6e}")
print("Whitened latent train mean (max abs):", float(jnp.max(jnp.abs(jnp.mean(TRAIN_Z, axis=0)))))
print("Whitened latent train variance range:",
      float(jnp.min(jnp.var(TRAIN_Z, axis=0))),
      float(jnp.max(jnp.var(TRAIN_Z, axis=0))))

# Visual reconstruction check.
plot_ids = np.arange(10)
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for col, idx in enumerate(plot_ids):
    axes[0, col].imshow(test_x_pixels_np[idx].reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axes[1, col].imshow(recon_test_np[idx].reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    axes[0, col].axis("off")
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("PCA-64")
plt.suptitle("Fixed PCA representation: original versus reconstructed test digits")
plt.tight_layout()
plt.show()


def sample_latent_rows(key, latent_array, batch_size):
    idx = jax.random.randint(key, (batch_size,), 0, latent_array.shape[0])
    return latent_array[idx]

## Cell 3 — VP schedule, residual score network, DSM Stage-I training, EMA

In [ ]:
# ============================================================================
# Cell 3 — VP schedule, residual score network, DSM Stage-I training, EMA
# ============================================================================

def beta_t(t):
    return CONFIG["BETA_MIN"] + t * (CONFIG["BETA_MAX"] - CONFIG["BETA_MIN"])


def integrated_beta(t):
    return (
        CONFIG["BETA_MIN"] * t
        + 0.5 * (CONFIG["BETA_MAX"] - CONFIG["BETA_MIN"]) * t * t
    )


def alpha_t(t):
    return jnp.exp(-0.5 * integrated_beta(t))


def sigma_t(t):
    return jnp.sqrt(jnp.maximum(1.0 - jnp.exp(-integrated_beta(t)), 1e-8))


def perturb_latents(z0, t, noise):
    a = alpha_t(t)[..., None]
    s = sigma_t(t)[..., None]
    return a * z0 + s * noise


class ResidualScoreMLP:
    """Pure-JAX residual MLP with a PyTorch-like class interface."""

    def __init__(self, latent_dim, time_dim, hidden_dim, n_blocks):
        if time_dim % 2 != 0:
            raise ValueError("TIME_EMBED_DIM must be even.")
        self.latent_dim = int(latent_dim)
        self.time_dim = int(time_dim)
        self.hidden_dim = int(hidden_dim)
        self.n_blocks = int(n_blocks)
        half = time_dim // 2
        self.time_frequencies = jnp.exp(
            jnp.linspace(jnp.log(1.0), jnp.log(1_000.0), half)
        ).astype(jnp.float32)

    def init(self, key):
        n_keys = 1 + 2 * self.n_blocks + 1
        keys = iter(jax.random.split(key, n_keys))
        params = {
            "input": {
                "W": glorot_uniform(next(keys), self.latent_dim + self.time_dim, self.hidden_dim),
                "b": jnp.zeros((self.hidden_dim,), dtype=jnp.float32),
            },
            "blocks": [],
        }
        for _ in range(self.n_blocks):
            params["blocks"].append({
                "W1": glorot_uniform(next(keys), self.hidden_dim, self.hidden_dim),
                "b1": jnp.zeros((self.hidden_dim,), dtype=jnp.float32),
                "W2": glorot_uniform(next(keys), self.hidden_dim, self.hidden_dim),
                "b2": jnp.zeros((self.hidden_dim,), dtype=jnp.float32),
            })
        # A very small output initialization stabilizes the beginning of DSM.
        out_key = next(keys)
        params["output"] = {
            "W": 1e-3 * glorot_uniform(out_key, self.hidden_dim, self.latent_dim),
            "b": jnp.zeros((self.latent_dim,), dtype=jnp.float32),
        }
        return params

    def time_embedding(self, t):
        angles = 2.0 * jnp.pi * t[..., None] * self.time_frequencies[None, :]
        return jnp.concatenate([jnp.sin(angles), jnp.cos(angles)], axis=-1)

    def apply(self, params, t, z):
        # This method expects t:(B,) and z:(B,n).
        te = self.time_embedding(t)
        h = jnp.concatenate([z, te], axis=-1)
        h = jax.nn.silu(h @ params["input"]["W"] + params["input"]["b"])
        for block in params["blocks"]:
            r = jax.nn.silu(h @ block["W1"] + block["b1"])
            r = r @ block["W2"] + block["b2"]
            h = (h + r) / jnp.sqrt(2.0)
        return h @ params["output"]["W"] + params["output"]["b"]


SCORE_MODEL = ResidualScoreMLP(
    latent_dim=N,
    time_dim=CONFIG["TIME_EMBED_DIM"],
    hidden_dim=CONFIG["SCORE_HIDDEN_DIM"],
    n_blocks=CONFIG["SCORE_RES_BLOCKS"],
)


def eps_predict(params, t, z):
    """Noise prediction supporting scalar or batched t/z."""
    if z.ndim == 1:
        return SCORE_MODEL.apply(params, t.reshape(1), z.reshape(1, N))[0]
    return SCORE_MODEL.apply(params, t, z)


def score_predict(params, t, z):
    eps_hat = eps_predict(params, t, z)
    s = sigma_t(t)
    if z.ndim == 1:
        return -eps_hat / s
    return -eps_hat / s[:, None]


def sample_dsm_draws(key, latent_array, batch_size):
    k_idx, k_t, k_noise, k_deq = jax.random.split(key, 4)
    z0 = sample_latent_rows(k_idx, latent_array, batch_size)
    if CONFIG["DEQUANTIZATION"]:
        # Small latent-space noise prevents exact duplicate points without changing
        # the fixed PCA representation or learning an encoder.
        z0 = z0 + (1.0 / 255.0) * jax.random.uniform(
            k_deq, z0.shape, minval=-1.0, maxval=1.0
        )
    t = jax.random.uniform(
        k_t,
        (batch_size,),
        minval=CONFIG["TIME_EPS"],
        maxval=1.0 - CONFIG["TIME_EPS"],
    )
    noise = jax.random.normal(k_noise, z0.shape)
    return z0, t, noise


def dsm_loss_with_draws(params, z0, t, noise):
    zt = perturb_latents(z0, t, noise)
    pred = eps_predict(params, t, zt)
    return jnp.mean(jnp.mean((pred - noise) ** 2, axis=-1))


@partial(jax.jit, static_argnames=("grad_clip", "ema_decay"))
def baseline_train_step(params, ema_params, opt_state, z0, t, noise, lr, grad_clip, ema_decay):
    loss, grads = jax.value_and_grad(dsm_loss_with_draws)(params, z0, t, noise)
    grads, grad_norm = clip_tree_by_global_norm(grads, grad_clip)
    params, opt_state = adam_update(params, grads, opt_state, lr)
    ema_params = ema_update(ema_params, params, ema_decay)
    return params, ema_params, opt_state, loss, grad_norm


def evaluate_dsm(params, latent_array, seed, n_batches=None, batch_size=None):
    n_batches = int(n_batches or CONFIG["BASELINE_VAL_BATCHES"])
    batch_size = int(batch_size or CONFIG["BASELINE_BATCH_SIZE"])
    values = []
    for i in range(n_batches):
        key = jax.random.fold_in(jax.random.PRNGKey(seed), i)
        z0, t, noise = sample_dsm_draws(key, latent_array, batch_size)
        values.append(float(dsm_loss_with_draws(params, z0, t, noise)))
    return float(np.mean(values))


def train_score_model(seed, updates=None, verbose=True):
    updates = int(updates or CONFIG["BASELINE_UPDATES"])
    init_key = jax.random.PRNGKey(seed)
    params = SCORE_MODEL.init(init_key)
    ema_params = tree_copy(params)
    opt_state = adam_init(params)
    history = []
    best_val = float("inf")
    best_ema = tree_copy(ema_params)
    start = time.time()

    for step in range(1, updates + 1):
        draw_key = jax.random.fold_in(jax.random.PRNGKey(seed + 10_000), step)
        z0, t, noise = sample_dsm_draws(
            draw_key, TRAIN_Z, CONFIG["BASELINE_BATCH_SIZE"]
        )
        params, ema_params, opt_state, loss, grad_norm = baseline_train_step(
            params,
            ema_params,
            opt_state,
            z0,
            t,
            noise,
            jnp.asarray(CONFIG["BASELINE_LR"], dtype=jnp.float32),
            grad_clip=float(CONFIG["BASELINE_GRAD_CLIP"]),
            ema_decay=float(CONFIG["BASELINE_EMA_DECAY"]),
        )

        if step == 1 or step % CONFIG["BASELINE_PRINT_EVERY"] == 0 or step == updates:
            val = evaluate_dsm(
                ema_params,
                VAL_Z,
                seed=seed + 20_000 + step,
            )
            history.append((step, float(loss), val, float(grad_norm)))
            if val < best_val:
                best_val = val
                best_ema = tree_copy(ema_params)
            if verbose:
                print(
                    f"[Stage I seed={seed}] step {step:6d}/{updates} | "
                    f"train DSM {float(loss):.6f} | val DSM {val:.6f} | "
                    f"grad {float(grad_norm):.3f}"
                )

    elapsed = time.time() - start
    return {
        "params": params,
        "ema_params": best_ema,
        "last_ema_params": ema_params,
        "best_val_dsm": best_val,
        "history": history,
        "seconds": elapsed,
    }


print("Training Stage-I 64D VP score model...")
STAGE1 = train_score_model(CONFIG["SEED"], verbose=True)
BASELINE_PARAMS = STAGE1["params"]
BASELINE_EMA_PARAMS = STAGE1["ema_params"]
BASELINE_VAL_DSM = evaluate_dsm(BASELINE_EMA_PARAMS, VAL_Z, seed=73_001)
BASELINE_TEST_DSM = evaluate_dsm(BASELINE_EMA_PARAMS, TEST_Z, seed=73_002)

print("Stage-I time:", format_seconds(STAGE1["seconds"]))
print(f"Stage-I validation DSM: {BASELINE_VAL_DSM:.6f}")
print(f"Stage-I test DSM      : {BASELINE_TEST_DSM:.6f}")

if STAGE1["history"]:
    hist = np.asarray(STAGE1["history"], dtype=np.float64)
    plt.figure(figsize=(7, 4))
    plt.plot(hist[:, 0], hist[:, 1], label="training minibatch DSM")
    plt.plot(hist[:, 0], hist[:, 2], label="validation DSM")
    plt.xlabel("update")
    plt.ylabel("noise-prediction MSE")
    plt.title("Stage-I score-model training")
    plt.grid(True)
    plt.legend()
    plt.show()

## Cell 4 — 64D affine projectable Gaeta basis library (R=434)

In [ ]:
# ============================================================================
# Cell 4 — 64D affine projectable Gaeta basis library (R=434)
# ============================================================================

# Each basis element has
#   tau(u), tau_u(u),
#   xi(u,y)=A(u)y+c(u), and xi_u(u,y)=A_u(u)y+c_u(u).
# All xi are affine in y, so the Ito term 0.5*gamma^2*Delta_y xi vanishes.

BASIS_NAMES = []
BASIS_FAMILIES = []
_tau_poly = []          # coefficients of 1, u, u^2
_tau_exp_amp = []
_tau_exp_rate = []
_A_base = []
_A_is_exp = []
_A_rate = []
_c_base = []
_c_is_exp = []
_c_rate = []

ZERO_A = np.zeros((N, N), dtype=np.float32)
ZERO_C = np.zeros((N,), dtype=np.float32)
EYE_N = np.eye(N, dtype=np.float32)


def add_affine_basis(
    name,
    family,
    tau_poly=(0.0, 0.0, 0.0),
    tau_exp_amp=0.0,
    tau_exp_rate=0.0,
    A_base=None,
    A_is_exp=False,
    A_rate=0.0,
    c_base=None,
    c_is_exp=False,
    c_rate=0.0,
):
    BASIS_NAMES.append(str(name))
    BASIS_FAMILIES.append(str(family))
    _tau_poly.append(np.asarray(tau_poly, dtype=np.float32))
    _tau_exp_amp.append(np.float32(tau_exp_amp))
    _tau_exp_rate.append(np.float32(tau_exp_rate))
    _A_base.append(np.asarray(ZERO_A if A_base is None else A_base, dtype=np.float32))
    _A_is_exp.append(bool(A_is_exp))
    _A_rate.append(np.float32(A_rate))
    _c_base.append(np.asarray(ZERO_C if c_base is None else c_base, dtype=np.float32))
    _c_is_exp.append(bool(c_is_exp))
    _c_rate.append(np.float32(c_rate))


# 1) Three time-only templates: d/du, u d/du, u^2 d/du.
add_affine_basis("time_translation", "time", tau_poly=(1.0, 0.0, 0.0))
add_affine_basis("time_linear_u", "time", tau_poly=(0.0, 1.0, 0.0))
add_affine_basis("time_quadratic_u2", "time", tau_poly=(0.0, 0.0, 1.0))

# 2) Coordinate translations.
for i in range(N):
    e = np.zeros((N,), dtype=np.float32)
    e[i] = 1.0
    add_affine_basis(f"translation_x{i:02d}", "translation", c_base=e)

# 3) Coordinate scalings x_i d/dx_i.
for i in range(N):
    A = np.zeros((N, N), dtype=np.float32)
    A[i, i] = 1.0
    add_affine_basis(f"coordinate_scaling_x{i:02d}", "scaling", A_base=A)

# 4) Lifted Brownian scaling 2u d/du + y.grad_y.
add_affine_basis(
    "global_brownian_scaling",
    "brownian_scaling",
    tau_poly=(0.0, 2.0, 0.0),
    A_base=EYE_N,
)

# 5) Component exponential translations e^{mu u} d/dx_i.
for mu in CONFIG["BASIS_MU_LIST"]:
    for i in range(N):
        e = np.zeros((N,), dtype=np.float32)
        e[i] = 1.0
        add_affine_basis(
            f"exp_translation_mu_{mu:+.2f}_x{i:02d}",
            "exp_translation",
            c_base=e,
            c_is_exp=True,
            c_rate=float(mu),
        )

# 6) Lifted projective templates e^{2mu u}(d/du + mu y.grad_y).
for mu in CONFIG["BASIS_MU_LIST"]:
    add_affine_basis(
        f"lifted_projective_mu_{mu:+.2f}",
        "lifted_projective",
        tau_exp_amp=1.0,
        tau_exp_rate=2.0 * float(mu),
        A_base=float(mu) * EYE_N,
        A_is_exp=True,
        A_rate=2.0 * float(mu),
    )


def hadamard_matrix(order):
    if order < 1 or (order & (order - 1)) != 0:
        raise ValueError("Hadamard order must be a positive power of two.")
    H = np.array([[1.0]], dtype=np.float32)
    while H.shape[0] < order:
        H = np.block([[H, H], [H, -H]]).astype(np.float32)
    return H


H64 = hadamard_matrix(N) / np.sqrt(float(N))
K_DIR = int(CONFIG["HADAMARD_DIRECTIONS"])
# Skip the constant row so that all selected directions carry spatial contrast.
HADAMARD_DIRS = H64[1:1 + K_DIR]

# 7) Low-rank rotations among eight deterministic Hadamard directions.
for a in range(K_DIR):
    for b in range(a + 1, K_DIR):
        va, vb = HADAMARD_DIRS[a], HADAMARD_DIRS[b]
        A = np.outer(va, vb) - np.outer(vb, va)
        add_affine_basis(
            f"hadamard_rotation_{a:02d}_{b:02d}",
            "hadamard_rotation",
            A_base=A,
        )

# 8) Adjacent bidirectional shears: (v_b.y)v_a.grad and reverse.
for a in range(K_DIR - 1):
    va, vb = HADAMARD_DIRS[a], HADAMARD_DIRS[a + 1]
    add_affine_basis(
        f"hadamard_shear_{a:02d}_from_{a+1:02d}",
        "hadamard_shear",
        A_base=np.outer(va, vb),
    )
    add_affine_basis(
        f"hadamard_shear_{a+1:02d}_from_{a:02d}",
        "hadamard_shear",
        A_base=np.outer(vb, va),
    )


TAU_POLY = jnp.asarray(np.stack(_tau_poly), dtype=jnp.float32)
TAU_EXP_AMP = jnp.asarray(np.asarray(_tau_exp_amp), dtype=jnp.float32)
TAU_EXP_RATE = jnp.asarray(np.asarray(_tau_exp_rate), dtype=jnp.float32)
A_BASE = jnp.asarray(np.stack(_A_base), dtype=jnp.float32)
A_IS_EXP = jnp.asarray(np.asarray(_A_is_exp), dtype=bool)
A_RATE = jnp.asarray(np.asarray(_A_rate), dtype=jnp.float32)
C_BASE = jnp.asarray(np.stack(_c_base), dtype=jnp.float32)
C_IS_EXP = jnp.asarray(np.asarray(_c_is_exp), dtype=bool)
C_RATE = jnp.asarray(np.asarray(_c_rate), dtype=jnp.float32)

R = len(BASIS_NAMES)
assert R == 434, f"Expected R=434, obtained R={R}."
print(f"Constructed affine projectable library with R={R} templates.")

family_counts = {}
for family in BASIS_FAMILIES:
    family_counts[family] = family_counts.get(family, 0) + 1
for family, count in family_counts.items():
    print(f"  {family:24s}: {count:3d}")


def safe_basis_exp(x):
    clip = float(CONFIG["BASIS_EXP_CLIP"])
    return jnp.exp(jnp.clip(x, -clip, clip))


@jax.jit
def eval_basis_affine(u):
    """Return tau,tau_u,A,A_u,c,c_u for all R basis elements at scalar u."""
    powers = jnp.stack([jnp.ones_like(u), u, u * u])
    dpowers = jnp.stack([jnp.zeros_like(u), jnp.ones_like(u), 2.0 * u])

    tau_poly = TAU_POLY @ powers
    tau_poly_u = TAU_POLY @ dpowers
    tau_exp = TAU_EXP_AMP * safe_basis_exp(TAU_EXP_RATE * u)
    tau_exp_u = TAU_EXP_RATE * tau_exp
    tau = tau_poly + tau_exp
    tau_u = tau_poly_u + tau_exp_u

    a_exp_scale = safe_basis_exp(A_RATE * u)
    a_scale = jnp.where(A_IS_EXP, a_exp_scale, 1.0)
    a_u_scale = jnp.where(A_IS_EXP, A_RATE * a_exp_scale, 0.0)
    A = A_BASE * a_scale[:, None, None]
    A_u = A_BASE * a_u_scale[:, None, None]

    c_exp_scale = safe_basis_exp(C_RATE * u)
    c_scale = jnp.where(C_IS_EXP, c_exp_scale, 1.0)
    c_u_scale = jnp.where(C_IS_EXP, C_RATE * c_exp_scale, 0.0)
    c = C_BASE * c_scale[:, None]
    c_u = C_BASE * c_u_scale[:, None]
    return tau, tau_u, A, A_u, c, c_u


def eval_generators_affine(W, u):
    """Combine basis templates using W:(m,R)."""
    tau_b, tau_u_b, A_b, A_u_b, c_b, c_u_b = eval_basis_affine(u)
    tau = W @ tau_b
    tau_u = W @ tau_u_b
    A = jnp.einsum("mr,rij->mij", W, A_b)
    A_u = jnp.einsum("mr,rij->mij", W, A_u_b)
    c = W @ c_b
    c_u = W @ c_u_b
    return tau, tau_u, A, A_u, c, c_u


def basis_values_at_point(u, y):
    tau, _, A, _, c, _ = eval_basis_affine(u)
    xi = jnp.einsum("rij,j->ri", A, y) + c
    return tau, xi

## Cell 5 — Reverse-SDE determining operator, Hutchinson/spectral discovery, affine Lie closure, adequacy gate, and m={44,54,64}

In [ ]:
# ============================================================================
# Cell 5 — Reverse-SDE determining operator, Hutchinson/spectral discovery,
#          affine Lie closure, adequacy gate, and m={44,54,64}
# ============================================================================

BETA_SLOPE = float(CONFIG["BETA_MAX"] - CONFIG["BETA_MIN"])


def reverse_gamma(u):
    t = 1.0 - u
    return jnp.sqrt(beta_t(t))


def reverse_gamma_u(u):
    # beta(1-u)' = -(beta_max-beta_min)
    return -0.5 * BETA_SLOPE / reverse_gamma(u)


def reverse_drift(score_params, u, y):
    """Generative-clock reverse drift, with u increasing from noise to data."""
    t = 1.0 - u
    b = beta_t(t)
    score = score_predict(score_params, t, y)
    return 0.5 * b * y + b * score


def sample_collocation_mixture(key, latent_array, n_points):
    """
    50% properly noised latent states, 25% jittered noised states,
    and 25% independent N(0,I) ambient points.
    """
    n_data = n_points // 2
    n_jitter = n_points // 4
    n_ambient = n_points - n_data - n_jitter
    k_data, k_jitter, k_ambient = jax.random.split(key, 3)

    def noised_points(k, count, add_jitter):
        k_idx, k_t, k_noise, k_extra = jax.random.split(k, 4)
        z0 = sample_latent_rows(k_idx, latent_array, count)
        t = jax.random.uniform(
            k_t,
            (count,),
            minval=CONFIG["TIME_EPS"],
            maxval=1.0 - CONFIG["TIME_EPS"],
        )
        y = perturb_latents(z0, t, jax.random.normal(k_noise, z0.shape))
        if add_jitter:
            y = y + 0.15 * jax.random.normal(k_extra, y.shape)
        return 1.0 - t, y

    u_data, y_data = noised_points(k_data, n_data, False)
    u_jitter, y_jitter = noised_points(k_jitter, n_jitter, True)

    k_u, k_y = jax.random.split(k_ambient)
    u_ambient = jax.random.uniform(
        k_u,
        (n_ambient,),
        minval=CONFIG["TIME_EPS"],
        maxval=1.0 - CONFIG["TIME_EPS"],
    )
    y_ambient = jax.random.normal(k_y, (n_ambient, N))

    u = jnp.concatenate([u_data, u_jitter, u_ambient], axis=0)
    y = jnp.concatenate([y_data, y_jitter, y_ambient], axis=0)
    perm_key = jax.random.fold_in(key, 991)
    perm = jax.random.permutation(perm_key, n_points)
    return u[perm], y[perm]


def basis_probe_features_one(score_params, u, y, v, w, q0, qx):
    """
    Scalar random projections whose outer products estimate:
      H = determining-residual Gram matrix,
      G = generator-magnitude Gram matrix.
    """
    b = reverse_drift(score_params, u, y)
    _, b_u = jax.jvp(
        lambda uu: reverse_drift(score_params, uu, y),
        (u,),
        (jnp.ones_like(u),),
    )
    jt_v = jax.grad(lambda yy: jnp.dot(reverse_drift(score_params, u, yy), v))(y)

    tau, tau_u, A, A_u, c, c_u = eval_basis_affine(u)
    xi = jnp.einsum("rij,j->ri", A, y) + c
    xi_u = jnp.einsum("rij,j->ri", A_u, y) + c_u

    v_dot_b = jnp.dot(v, b)
    v_dot_bu = jnp.dot(v, b_u)
    drift_proj = (
        xi_u @ v
        + jnp.einsum("i,rij,j->r", v, A, b)
        - xi @ jt_v
        - tau * v_dot_bu
        - tau_u * v_dot_b
    ) / jnp.sqrt(float(N))

    gamma = reverse_gamma(u)
    gamma_u = reverse_gamma_u(u)
    scalar_coeff = tau * gamma_u + 0.5 * gamma * tau_u
    diff_proj = (
        gamma * jnp.einsum("i,rij,j->r", v, A, w)
        - scalar_coeff * jnp.dot(v, w)
    ) / jnp.sqrt(float(N))

    generator_proj = q0 * tau + (xi @ qx) / jnp.sqrt(float(N))
    return drift_proj, diff_proj, generator_proj


def probe_features_for_state(score_params, u, y, vs, ws, q0s, qxs):
    return jax.vmap(
        lambda v, w, q0, qx: basis_probe_features_one(
            score_params, u, y, v, w, q0, qx
        )
    )(vs, ws, q0s, qxs)


@partial(jax.jit, static_argnames=("n_probes",))
def compute_feature_arrays(score_params, u, y, key, n_probes):
    s = u.shape[0]
    k_v, k_w, k_q0, k_qx = jax.random.split(key, 4)
    vs = rademacher(k_v, (s, n_probes, N))
    ws = rademacher(k_w, (s, n_probes, N))
    q0s = rademacher(k_q0, (s, n_probes))
    qxs = rademacher(k_qx, (s, n_probes, N))
    return jax.vmap(
        lambda uu, yy, vv, ww, qq0, qqx: probe_features_for_state(
            score_params, uu, yy, vv, ww, qq0, qqx
        )
    )(u, y, vs, ws, q0s, qxs)


def grams_from_feature_arrays(drift_feat, diff_feat, generator_feat, sample_indices=None):
    if sample_indices is not None:
        drift_feat = drift_feat[sample_indices]
        diff_feat = diff_feat[sample_indices]
        generator_feat = generator_feat[sample_indices]
    d = drift_feat.reshape(-1, R)
    s = diff_feat.reshape(-1, R)
    g = generator_feat.reshape(-1, R)
    H = (d.T @ d + s.T @ s) / float(d.shape[0])
    G = (g.T @ g) / float(g.shape[0])
    H = 0.5 * (H + H.T)
    G = 0.5 * (G + G.T)
    return H, G


def generalized_low_residual_directions(H, G, m, ridge=None):
    ridge = float(ridge or CONFIG["DISCOVERY_SPECTRAL_RIDGE"])
    eye = jnp.eye(R, dtype=jnp.float32)
    G_sym = 0.5 * (G + G.T)
    H_sym = 0.5 * (H + H.T)
    # A relative ridge prevents nearly-zero generator-magnitude directions from
    # acquiring enormous coefficients in the generalized eigenproblem.
    g_scale = jnp.maximum(jnp.mean(jnp.diag(G_sym)), 1e-6)
    h_scale = jnp.maximum(jnp.mean(jnp.diag(H_sym)), 1e-6)
    G_reg = G_sym + ridge * g_scale * eye
    H_reg = H_sym + (1e-7 * h_scale) * eye

    g_eval, g_evec = jnp.linalg.eigh(G_reg)
    g_floor = ridge * jnp.maximum(jnp.max(g_eval), 1e-6)
    g_eval = jnp.maximum(g_eval, g_floor)
    G_inv_half = (g_evec * (1.0 / jnp.sqrt(g_eval))[None, :]) @ g_evec.T
    M = G_inv_half @ H_reg @ G_inv_half
    M = 0.5 * (M + M.T)
    evals, evecs = jnp.linalg.eigh(M)
    idx = jnp.argsort(evals)[:m]
    W = (G_inv_half @ evecs[:, idx]).T
    # Re-orthonormalize numerically in the G metric.
    gram = W @ G_reg @ W.T
    ge, gv = jnp.linalg.eigh(0.5 * (gram + gram.T))
    W = ((gv * (1.0 / jnp.sqrt(jnp.maximum(ge, ridge)))[None, :]) @ gv.T) @ W
    return W, evals[idx]


def normalized_subspace_score(W, H, G):
    num = jnp.maximum(jnp.trace(W @ H @ W.T), 0.0)
    den = jnp.maximum(jnp.trace(W @ G @ W.T), 1e-12)
    return num / den


def sample_random_g_orthonormal_subspace(key, G, m):
    raw = jax.random.normal(key, (m, R))
    ridge = float(CONFIG["DISCOVERY_SPECTRAL_RIDGE"])
    g_scale = jnp.maximum(jnp.mean(jnp.diag(G)), 1e-6)
    gram = raw @ (G + ridge * g_scale * jnp.eye(R)) @ raw.T
    ge, gv = jnp.linalg.eigh(0.5 * (gram + gram.T))
    inv_half = (gv * (1.0 / jnp.sqrt(jnp.maximum(ge, ridge)))[None, :]) @ gv.T
    return inv_half @ raw


def choose_pair_subset(m, max_pairs, seed):
    all_pairs = [(a, b) for a in range(m) for b in range(a + 1, m)]
    rng = np.random.default_rng(seed)
    if len(all_pairs) > max_pairs:
        chosen = rng.choice(len(all_pairs), size=max_pairs, replace=False)
        all_pairs = [all_pairs[i] for i in chosen]
    i = jnp.asarray([p[0] for p in all_pairs], dtype=jnp.int32)
    j = jnp.asarray([p[1] for p in all_pairs], dtype=jnp.int32)
    return i, j


def closure_loss_affine(W, closure_u, closure_y, pair_i, pair_j):
    def one_time(u):
        return eval_generators_affine(W, u)

    tau, tau_u, A, A_u, c, c_u = jax.vmap(one_time)(closure_u)
    xi = jnp.einsum("smij,sj->smi", A, closure_y) + c
    Xvals = jnp.concatenate([tau[..., None], xi], axis=-1)  # (S,m,n+1)

    ta, tb = tau[:, pair_i], tau[:, pair_j]
    tua, tub = tau_u[:, pair_i], tau_u[:, pair_j]
    Aa, Ab = A[:, pair_i], A[:, pair_j]
    Aua, Aub = A_u[:, pair_i], A_u[:, pair_j]
    ca, cb = c[:, pair_i], c[:, pair_j]
    cua, cub = c_u[:, pair_i], c_u[:, pair_j]

    bracket_tau = ta * tub - tb * tua
    bracket_A = (
        ta[..., None, None] * Aub
        - tb[..., None, None] * Aua
        + jnp.einsum("spij,spjk->spik", Ab, Aa)
        - jnp.einsum("spij,spjk->spik", Aa, Ab)
    )
    bracket_c = (
        ta[..., None] * cub
        - tb[..., None] * cua
        + jnp.einsum("spij,spj->spi", Ab, ca)
        - jnp.einsum("spij,spj->spi", Aa, cb)
    )
    bracket_xi = jnp.einsum("spij,sj->spi", bracket_A, closure_y) + bracket_c
    Bvals = jnp.concatenate([bracket_tau[..., None], bracket_xi], axis=-1)

    # One global least-squares projection gives constant structure constants.
    Xdesign = jnp.transpose(Xvals, (0, 2, 1)).reshape(-1, W.shape[0])
    Bdesign = jnp.transpose(Bvals, (0, 2, 1)).reshape(-1, pair_i.shape[0])
    ridge = float(CONFIG["DISCOVERY_CLOSURE_RIDGE"])
    lhs = Xdesign.T @ Xdesign + ridge * jnp.eye(W.shape[0])
    rhs = Xdesign.T @ Bdesign
    coeff = jnp.linalg.solve(lhs, rhs)
    residual = Bdesign - Xdesign @ coeff
    # Relative closure defect is scale-stable across m and across libraries.
    numerator = jnp.mean(residual * residual)
    denominator = jax.lax.stop_gradient(jnp.mean(Bdesign * Bdesign) + 1e-6)
    return numerator / denominator


def stage2_objective(W, H_train, G_train, closure_u, closure_y, pair_i, pair_j):
    m = W.shape[0]
    det = jnp.maximum(jnp.trace(W @ H_train @ W.T) / float(m), 0.0)
    gen_gram = W @ G_train @ W.T
    ind = jnp.mean((gen_gram - jnp.eye(m)) ** 2)
    diag = jnp.diag(gen_gram)
    norm = jnp.mean(jax.nn.relu(CONFIG["DISCOVERY_MIN_NORM"] - diag) ** 2)
    close = closure_loss_affine(W, closure_u, closure_y, pair_i, pair_j)
    total = (
        CONFIG["DISCOVERY_LAMBDA_DET"] * det
        + CONFIG["DISCOVERY_LAMBDA_CLOSE"] * close
        + CONFIG["DISCOVERY_LAMBDA_IND"] * ind
        + CONFIG["DISCOVERY_LAMBDA_NORM"] * norm
    )
    return total, {"det": det, "close": close, "ind": ind, "norm": norm}


@partial(jax.jit, static_argnames=("grad_clip",))
def stage2_step(W, opt_state, H_train, G_train, closure_u, closure_y, pair_i, pair_j, lr, grad_clip):
    (loss, aux), grad = jax.value_and_grad(stage2_objective, has_aux=True)(
        W, H_train, G_train, closure_u, closure_y, pair_i, pair_j
    )
    grad, grad_norm = clip_tree_by_global_norm(grad, grad_clip)
    W, opt_state = adam_update(W, grad, opt_state, lr)
    return W, opt_state, loss, aux, grad_norm


def exact_generator_residual_one(score_params, W, u, y):
    b = reverse_drift(score_params, u, y)
    _, b_u = jax.jvp(
        lambda uu: reverse_drift(score_params, uu, y),
        (u,),
        (jnp.ones_like(u),),
    )
    Jb = jax.jacrev(lambda yy: reverse_drift(score_params, u, yy))(y)
    tau, tau_u, A, A_u, c, c_u = eval_generators_affine(W, u)
    xi = jnp.einsum("mij,j->mi", A, y) + c
    xi_u = jnp.einsum("mij,j->mi", A_u, y) + c_u
    Rb = (
        xi_u
        + jnp.einsum("mij,j->mi", A, b)
        - jnp.einsum("ij,mj->mi", Jb, xi)
        - tau[:, None] * b_u[None, :]
        - tau_u[:, None] * b[None, :]
    )
    gamma = reverse_gamma(u)
    gamma_u = reverse_gamma_u(u)
    scalar = tau * gamma_u + 0.5 * gamma * tau_u
    Rs = gamma * A - scalar[:, None, None] * jnp.eye(N)[None, :, :]
    return jnp.mean(jnp.sum(Rb * Rb, axis=-1) / N), jnp.mean(
        jnp.sum(Rs * Rs, axis=(-2, -1)) / N
    )


def exact_heldout_residual(score_params, W, u, y):
    drift, diff = jax.vmap(
        lambda uu, yy: exact_generator_residual_one(score_params, W, uu, yy)
    )(u, y)
    return float(jnp.mean(drift)), float(jnp.mean(diff))


def bootstrap_score_cv(W, drift_feat, diff_feat, generator_feat, seed):
    rng = np.random.default_rng(seed)
    s = drift_feat.shape[0]
    scores = []
    for _ in range(CONFIG["ADEQUACY_BOOTSTRAPS"]):
        idx = jnp.asarray(rng.integers(0, s, size=s), dtype=jnp.int32)
        H_b, G_b = grams_from_feature_arrays(
            drift_feat, diff_feat, generator_feat, sample_indices=idx
        )
        scores.append(float(normalized_subspace_score(W, H_b, G_b)))
    mean = float(np.mean(scores))
    cv = float(np.std(scores) / max(abs(mean), 1e-12))
    return mean, cv, scores


def build_discovery_statistics(score_params, seed):
    train_u, train_y = sample_collocation_mixture(
        jax.random.PRNGKey(seed + 1), TRAIN_Z, CONFIG["DISCOVERY_TRAIN_POINTS"]
    )
    val_u, val_y = sample_collocation_mixture(
        jax.random.PRNGKey(seed + 2), VAL_Z, CONFIG["DISCOVERY_VAL_POINTS"]
    )
    train_features = compute_feature_arrays(
        score_params,
        train_u,
        train_y,
        jax.random.PRNGKey(seed + 3),
        n_probes=int(CONFIG["DISCOVERY_PROBES"]),
    )
    val_features = compute_feature_arrays(
        score_params,
        val_u,
        val_y,
        jax.random.PRNGKey(seed + 4),
        n_probes=int(CONFIG["DISCOVERY_PROBES"]),
    )
    H_train, G_train = grams_from_feature_arrays(*train_features)
    H_val, G_val = grams_from_feature_arrays(*val_features)
    return {
        "train_u": train_u,
        "train_y": train_y,
        "val_u": val_u,
        "val_y": val_y,
        "train_features": train_features,
        "val_features": val_features,
        "H_train": H_train,
        "G_train": G_train,
        "H_val": H_val,
        "G_val": G_val,
    }


def discover_for_m(score_params, stats, m, seed, verbose=True):
    W, spectral_evals = generalized_low_residual_directions(
        stats["H_train"], stats["G_train"], m
    )

    closure_u, closure_y = sample_collocation_mixture(
        jax.random.PRNGKey(seed + 100 + m),
        TRAIN_Z,
        CONFIG["DISCOVERY_CLOSURE_POINTS"],
    )
    pair_i, pair_j = choose_pair_subset(
        m, CONFIG["DISCOVERY_CLOSURE_PAIRS"], seed + 200 + m
    )
    opt_state = adam_init(W)
    history = []

    for step in range(1, CONFIG["DISCOVERY_ADAM_STEPS"] + 1):
        W, opt_state, loss, aux, grad_norm = stage2_step(
            W,
            opt_state,
            stats["H_train"],
            stats["G_train"],
            closure_u,
            closure_y,
            pair_i,
            pair_j,
            jnp.asarray(CONFIG["DISCOVERY_ADAM_LR"], dtype=jnp.float32),
            grad_clip=5.0,
        )
        if step == 1 or step % CONFIG["DISCOVERY_PRINT_EVERY"] == 0 or step == CONFIG["DISCOVERY_ADAM_STEPS"]:
            row = {
                "step": step,
                "total": float(loss),
                "det": float(aux["det"]),
                "close": float(aux["close"]),
                "ind": float(aux["ind"]),
                "norm": float(aux["norm"]),
                "grad": float(grad_norm),
            }
            history.append(row)
            if verbose:
                print(
                    f"[Stage II m={m}] step {step:4d} | total {row['total']:.4e} | "
                    f"det {row['det']:.4e} | close {row['close']:.4e} | "
                    f"ind {row['ind']:.4e}"
                )

    train_score = float(normalized_subspace_score(W, stats["H_train"], stats["G_train"]))
    val_score = float(normalized_subspace_score(W, stats["H_val"], stats["G_val"]))
    val_train_ratio = val_score / max(train_score, 1e-12)

    random_scores = []
    for r_idx in range(CONFIG["ADEQUACY_RANDOM_SUBSPACES"]):
        Wr = sample_random_g_orthonormal_subspace(
            jax.random.PRNGKey(seed + 10_000 + 100 * m + r_idx),
            stats["G_val"],
            m,
        )
        random_scores.append(float(normalized_subspace_score(Wr, stats["H_val"], stats["G_val"])))
    random_threshold = float(np.quantile(random_scores, CONFIG["ADEQUACY_RANDOM_QUANTILE"]))
    random_median = float(np.median(random_scores))

    boot_mean, boot_cv, boot_scores = bootstrap_score_cv(
        W,
        *stats["val_features"],
        seed=seed + 20_000 + m,
    )

    passes_low = val_score < random_threshold
    passes_inflation = val_train_ratio <= CONFIG["ADEQUACY_MAX_VAL_TRAIN_RATIO"]
    passes_bootstrap = boot_cv <= CONFIG["ADEQUACY_MAX_BOOTSTRAP_CV"]
    accepted = bool(passes_low and passes_inflation and passes_bootstrap)

    exact_n = min(CONFIG["ADEQUACY_EXACT_POINTS"], stats["val_u"].shape[0])
    exact_drift, exact_diff = exact_heldout_residual(
        score_params,
        W,
        stats["val_u"][:exact_n],
        stats["val_y"][:exact_n],
    )

    diagnostics = {
        "m": int(m),
        "accepted": accepted,
        "train_score": train_score,
        "val_score": val_score,
        "val_train_ratio": val_train_ratio,
        "random_threshold_q05": random_threshold,
        "random_median": random_median,
        "bootstrap_mean": boot_mean,
        "bootstrap_cv": boot_cv,
        "bootstrap_scores": boot_scores,
        "exact_drift_residual": exact_drift,
        "exact_diffusion_residual": exact_diff,
        "spectral_eval_mean": float(jnp.mean(spectral_evals)),
        "passes_low_residual": bool(passes_low),
        "passes_train_val": bool(passes_inflation),
        "passes_bootstrap": bool(passes_bootstrap),
    }
    return {"W": W, "diagnostics": diagnostics, "history": history}


def discover_generator_branches(score_params, m_values, seed, verbose=True):
    start = time.time()
    stats = build_discovery_statistics(score_params, seed)
    branches = {}
    for m in m_values:
        branches[int(m)] = discover_for_m(
            score_params, stats, int(m), seed, verbose=verbose
        )
    elapsed = time.time() - start
    return branches, stats, elapsed


print("Building Stage-II residual and generator Gram matrices...")
DISCOVERED, DISCOVERY_STATS, DISCOVERY_SECONDS = discover_generator_branches(
    BASELINE_EMA_PARAMS,
    CONFIG["GENERATOR_COUNTS"],
    seed=CONFIG["SEED"] + 30_000,
    verbose=True,
)

print("Stage-II total time:", format_seconds(DISCOVERY_SECONDS))
print("\nData-only adequacy summary")
print("m  gate      train-q       val-q    val/train    random-q05   boot-CV   exact drift/diff")
for m in CONFIG["GENERATOR_COUNTS"]:
    d = DISCOVERED[m]["diagnostics"]
    print(
        f"{m:2d} {'ACCEPT' if d['accepted'] else 'ABSTAIN':8s} "
        f"{d['train_score']:11.4e} {d['val_score']:11.4e} "
        f"{d['val_train_ratio']:10.3f} {d['random_threshold_q05']:11.4e} "
        f"{d['bootstrap_cv']:8.3f} "
        f"{d['exact_drift_residual']:.3e}/{d['exact_diffusion_residual']:.3e}"
    )



## Cell 6 — Lambda pilot, matched score-only continuation, and Gaeta-Lie Stage III

In [ ]:
# ============================================================================
# Cell 6 — Lambda pilot, matched score-only continuation, and Gaeta-Lie Stage III
# ============================================================================

def generator_drift_probe_one(score_params, W, u, y, v):
    b = reverse_drift(score_params, u, y)
    _, b_u = jax.jvp(
        lambda uu: reverse_drift(score_params, uu, y),
        (u,),
        (jnp.ones_like(u),),
    )
    jt_v = jax.grad(lambda yy: jnp.dot(reverse_drift(score_params, u, yy), v))(y)
    tau, tau_u, A, A_u, c, c_u = eval_generators_affine(W, u)
    xi = jnp.einsum("mij,j->mi", A, y) + c
    xi_u = jnp.einsum("mij,j->mi", A_u, y) + c_u
    projected = (
        xi_u @ v
        + jnp.einsum("i,mij,j->m", v, A, b)
        - xi @ jt_v
        - tau * jnp.dot(v, b_u)
        - tau_u * jnp.dot(v, b)
    )
    return jnp.mean(projected * projected) / float(N)


def generator_drift_probe_state(score_params, W, u, y, probes):
    return jnp.mean(
        jax.vmap(lambda v: generator_drift_probe_one(score_params, W, u, y, v))(probes)
    )


def generator_hutchinson_drift_loss(score_params, W, u, y, probes):
    return jnp.mean(
        jax.vmap(
            lambda uu, yy, vv: generator_drift_probe_state(score_params, W, uu, yy, vv)
        )(u, y, probes)
    )


def make_refinement_draws(seed, step, sym_points, n_probes):
    base = jax.random.PRNGKey(seed)
    data_key = jax.random.fold_in(base, 2 * step)
    sym_key = jax.random.fold_in(base, 2 * step + 1)
    z0, t, noise = sample_dsm_draws(data_key, TRAIN_Z, CONFIG["REFINE_BATCH_SIZE"])
    u, y = sample_collocation_mixture(sym_key, TRAIN_Z, sym_points)
    probe_key = jax.random.fold_in(sym_key, 777)
    probes = rademacher(probe_key, (sym_points, n_probes, N))
    return z0, t, noise, u, y, probes


def combined_refine_loss(params, W, z0, t, noise, u, y, probes, lambda_eff, gl_scale):
    dsm = dsm_loss_with_draws(params, z0, t, noise)
    det_raw = generator_hutchinson_drift_loss(params, W, u, y, probes)
    det_normalized = det_raw / jnp.maximum(gl_scale, 1e-12)
    total = dsm + lambda_eff * det_normalized
    return total, {"dsm": dsm, "det_raw": det_raw, "det_norm": det_normalized}


@partial(jax.jit, static_argnames=("grad_clip", "ema_decay"))
def gl_refine_step(
    params,
    ema_params,
    opt_state,
    W,
    z0,
    t,
    noise,
    u,
    y,
    probes,
    lambda_eff,
    gl_scale,
    lr,
    grad_clip,
    ema_decay,
):
    (loss, aux), grads = jax.value_and_grad(combined_refine_loss, has_aux=True)(
        params, W, z0, t, noise, u, y, probes, lambda_eff, gl_scale
    )
    grads, grad_norm = clip_tree_by_global_norm(grads, grad_clip)
    params, opt_state = adam_update(params, grads, opt_state, lr)
    ema_params = ema_update(ema_params, params, ema_decay)
    return params, ema_params, opt_state, loss, aux, grad_norm


@partial(jax.jit, static_argnames=("grad_clip", "ema_decay"))
def continuation_step(
    params, ema_params, opt_state, z0, t, noise, lr, grad_clip, ema_decay
):
    loss, grads = jax.value_and_grad(dsm_loss_with_draws)(params, z0, t, noise)
    grads, grad_norm = clip_tree_by_global_norm(grads, grad_clip)
    params, opt_state = adam_update(params, grads, opt_state, lr)
    ema_params = ema_update(ema_params, params, ema_decay)
    return params, ema_params, opt_state, loss, grad_norm


def calibration_gl_scale(score_params, W, seed):
    _, _, _, u, y, probes = make_refinement_draws(
        seed,
        step=0,
        sym_points=CONFIG["REFINE_SYM_POINTS"],
        n_probes=CONFIG["REFINE_HUTCHINSON_PROBES"],
    )
    return float(generator_hutchinson_drift_loss(score_params, W, u, y, probes))


def evaluate_generator_drift_loss(score_params, W, latent_array, seed, points=64):
    u, y = sample_collocation_mixture(jax.random.PRNGKey(seed), latent_array, points)
    probes = rademacher(
        jax.random.PRNGKey(seed + 1),
        (points, CONFIG["REFINE_HUTCHINSON_PROBES"], N),
    )
    return float(generator_hutchinson_drift_loss(score_params, W, u, y, probes))


def train_continuation(start_params, seed, updates, verbose=True):
    params = tree_copy(start_params)
    ema_params = tree_copy(start_params)
    opt_state = adam_init(params)
    history = []
    start = time.time()
    for step in range(1, updates + 1):
        z0, t, noise, _, _, _ = make_refinement_draws(
            seed,
            step,
            sym_points=CONFIG["REFINE_SYM_POINTS"],
            n_probes=CONFIG["REFINE_HUTCHINSON_PROBES"],
        )
        params, ema_params, opt_state, loss, grad_norm = continuation_step(
            params,
            ema_params,
            opt_state,
            z0,
            t,
            noise,
            jnp.asarray(CONFIG["REFINE_LR"], dtype=jnp.float32),
            grad_clip=float(CONFIG["REFINE_GRAD_CLIP"]),
            ema_decay=float(CONFIG["REFINE_EMA_DECAY"]),
        )
        if step == 1 or step % CONFIG["REFINE_PRINT_EVERY"] == 0 or step == updates:
            history.append((step, float(loss), float(grad_norm)))
            if verbose:
                print(
                    f"[Continuation] step {step:4d}/{updates} | DSM {float(loss):.6f} | "
                    f"grad {float(grad_norm):.3f}"
                )
    return {
        "params": params,
        "ema_params": ema_params,
        "history": history,
        "seconds": time.time() - start,
    }


def train_gl_branch(start_params, W, lambda_gl, seed, updates, verbose=True):
    params = tree_copy(start_params)
    ema_params = tree_copy(start_params)
    opt_state = adam_init(params)
    gl_scale = calibration_gl_scale(start_params, W, seed + 7_000)
    history = []
    start = time.time()

    warmup_steps = max(1, int(CONFIG["REFINE_WARMUP_FRACTION"] * updates))
    for step in range(1, updates + 1):
        z0, t, noise, u, y, probes = make_refinement_draws(
            seed,
            step,
            sym_points=CONFIG["REFINE_SYM_POINTS"],
            n_probes=CONFIG["REFINE_HUTCHINSON_PROBES"],
        )
        warm = min(1.0, step / float(warmup_steps))
        lambda_eff = jnp.asarray(lambda_gl * warm, dtype=jnp.float32)
        params, ema_params, opt_state, loss, aux, grad_norm = gl_refine_step(
            params,
            ema_params,
            opt_state,
            W,
            z0,
            t,
            noise,
            u,
            y,
            probes,
            lambda_eff,
            jnp.asarray(gl_scale, dtype=jnp.float32),
            jnp.asarray(CONFIG["REFINE_LR"], dtype=jnp.float32),
            grad_clip=float(CONFIG["REFINE_GRAD_CLIP"]),
            ema_decay=float(CONFIG["REFINE_EMA_DECAY"]),
        )
        if step == 1 or step % CONFIG["REFINE_PRINT_EVERY"] == 0 or step == updates:
            row = {
                "step": step,
                "total": float(loss),
                "dsm": float(aux["dsm"]),
                "det_raw": float(aux["det_raw"]),
                "det_norm": float(aux["det_norm"]),
                "grad": float(grad_norm),
                "lambda_eff": float(lambda_eff),
            }
            history.append(row)
            if verbose:
                print(
                    f"[Gaeta-Lie m={W.shape[0]}] step {step:4d}/{updates} | "
                    f"total {row['total']:.6f} | DSM {row['dsm']:.6f} | "
                    f"det/raw {row['det_raw']:.3e} | lambda {row['lambda_eff']:.3g}"
                )
    return {
        "params": params,
        "ema_params": ema_params,
        "history": history,
        "gl_scale": gl_scale,
        "seconds": time.time() - start,
    }


def select_lambda_with_pilot(start_params, discovered, seed):
    pilot_m = int(CONFIG["PILOT_M"])
    branch = discovered[pilot_m]
    if not branch["diagnostics"]["accepted"]:
        print(f"Pilot library gate abstained at m={pilot_m}; lambda selection returns 0.")
        return 0.0, []

    W = branch["W"]
    base_det = evaluate_generator_drift_loss(
        start_params, W, VAL_Z, seed + 400, points=64
    )
    rows = []
    for idx, lam in enumerate(CONFIG["PILOT_LAMBDAS"]):
        pilot = train_gl_branch(
            start_params,
            W,
            float(lam),
            seed=seed,
            updates=CONFIG["PILOT_UPDATES"],
            verbose=False,
        )
        val_dsm = evaluate_dsm(
            pilot["ema_params"], VAL_Z, seed=seed + 1_000 + idx,
            n_batches=max(2, CONFIG["BASELINE_VAL_BATCHES"] // 2),
        )
        val_det = evaluate_generator_drift_loss(
            pilot["ema_params"], W, VAL_Z, seed + 2_000 + idx, points=64
        )
        criterion = (
            val_dsm / max(BASELINE_VAL_DSM, 1e-12)
            + CONFIG["PILOT_VALIDATION_WEIGHT_DET"]
            * val_det / max(base_det, 1e-12)
        )
        rows.append({
            "lambda": float(lam),
            "val_dsm": val_dsm,
            "val_det": val_det,
            "criterion": criterion,
        })
        print(
            f"[Lambda pilot m={pilot_m}] lambda={lam:.3g} | "
            f"val DSM={val_dsm:.6f} | val det={val_det:.3e} | "
            f"criterion={criterion:.6f}"
        )
    selected = min(rows, key=lambda r: r["criterion"])["lambda"]
    print("Selected shared lambda_GL:", selected)
    return float(selected), rows


SELECTED_LAMBDA_GL, LAMBDA_PILOT_ROWS = select_lambda_with_pilot(
    BASELINE_EMA_PARAMS,
    DISCOVERED,
    seed=CONFIG["SEED"] + 40_000,
)

# All branches use the same deterministic sequence of DSM and collocation draws.
MATCHED_REFINE_SEED = CONFIG["SEED"] + 50_000
print("\nTraining matched DSM-only continuation...")
CONTINUATION = train_continuation(
    BASELINE_EMA_PARAMS,
    seed=MATCHED_REFINE_SEED,
    updates=CONFIG["REFINE_UPDATES"],
    verbose=True,
)

REFINED = {}
for m in CONFIG["GENERATOR_COUNTS"]:
    info = DISCOVERED[m]
    if not info["diagnostics"]["accepted"] or SELECTED_LAMBDA_GL <= 0.0:
        print(f"Skipping Stage III for m={m}: adequacy gate abstained or lambda=0.")
        REFINED[m] = {
            "skipped": True,
            "ema_params": tree_copy(BASELINE_EMA_PARAMS),
            "params": tree_copy(BASELINE_EMA_PARAMS),
            "seconds": 0.0,
            "gl_scale": float("nan"),
            "history": [],
        }
    else:
        print(f"\nTraining Gaeta-Lie Stage III for m={m}...")
        branch = train_gl_branch(
            BASELINE_EMA_PARAMS,
            info["W"],
            SELECTED_LAMBDA_GL,
            seed=MATCHED_REFINE_SEED,
            updates=CONFIG["REFINE_UPDATES"],
            verbose=True,
        )
        branch["skipped"] = False
        REFINED[m] = branch


MODEL_PARAMS = {
    "Stage-I": BASELINE_EMA_PARAMS,
    "Continuation": CONTINUATION["ema_params"],
}
for m in CONFIG["GENERATOR_COUNTS"]:
    if not REFINED[m]["skipped"]:
        MODEL_PARAMS[f"Gaeta-Lie m={m}"] = REFINED[m]["ema_params"]

print("\nPost-refinement DSM and determining-residual diagnostics")
print("model                 val DSM      test DSM     matching-generator drift residual")
POST_REFINE_DIAGNOSTICS = {}
for name, params in MODEL_PARAMS.items():
    val_dsm = evaluate_dsm(params, VAL_Z, seed=91_001)
    test_dsm = evaluate_dsm(params, TEST_Z, seed=91_002)
    if name.startswith("Gaeta-Lie"):
        m = int(name.split("=")[-1])
        det = evaluate_generator_drift_loss(
            params, DISCOVERED[m]["W"], VAL_Z, seed=91_003 + m, points=64
        )
    else:
        # Report residual against the pilot m=54 generators for common reference.
        det = evaluate_generator_drift_loss(
            params,
            DISCOVERED[CONFIG["PILOT_M"]]["W"],
            VAL_Z,
            seed=91_100,
            points=64,
        )
    POST_REFINE_DIAGNOSTICS[name] = {
        "val_dsm": val_dsm,
        "test_dsm": test_dsm,
        "val_det": det,
    }
    print(f"{name:21s} {val_dsm:11.6f} {test_dsm:13.6f} {det:18.4e}")

# For each accepted m, compare Stage-I, continuation, and refinement against
# exactly the same frozen m-generator set and exactly the same validation probes.
BRANCH_DET_DIAGNOSTICS = {}
for m in CONFIG["GENERATOR_COUNTS"]:
    gl_name = f"Gaeta-Lie m={m}"
    if gl_name not in MODEL_PARAMS:
        continue
    common_seed = 92_000 + m
    Wm = DISCOVERED[m]["W"]
    before = evaluate_generator_drift_loss(
        BASELINE_EMA_PARAMS, Wm, VAL_Z, seed=common_seed, points=64
    )
    continuation_det = evaluate_generator_drift_loss(
        CONTINUATION["ema_params"], Wm, VAL_Z, seed=common_seed, points=64
    )
    after = evaluate_generator_drift_loss(
        REFINED[m]["ema_params"], Wm, VAL_Z, seed=common_seed, points=64
    )
    BRANCH_DET_DIAGNOSTICS[m] = {
        "stage1": before,
        "continuation": continuation_det,
        "gaeta_lie": after,
        "reduction_vs_stage1_pct": 100.0 * (before - after) / max(before, 1e-12),
    }

## Cell 7 — Reverse-SDE sampling, frozen JAX evaluator, generative metrics, Phase-A m selection, optional three-seed confirmation, final report

In [ ]:
# ============================================================================
# Cell 7 — Reverse-SDE sampling, frozen JAX evaluator, generative metrics,
#          Phase-A m selection, optional three-seed confirmation, final report
# ============================================================================

class MNISTEvaluatorMLP:
    """Pure-JAX classifier; its penultimate activations define MNIST-FD features."""

    def __init__(self, in_dim=784, h1=256, feature_dim=128, n_classes=10):
        self.in_dim = in_dim
        self.h1 = h1
        self.feature_dim = feature_dim
        self.n_classes = n_classes

    def init(self, key):
        k1, k2, k3 = jax.random.split(key, 3)
        return {
            "l1": {"W": glorot_uniform(k1, self.in_dim, self.h1),
                   "b": jnp.zeros((self.h1,), dtype=jnp.float32)},
            "l2": {"W": glorot_uniform(k2, self.h1, self.feature_dim),
                   "b": jnp.zeros((self.feature_dim,), dtype=jnp.float32)},
            "out": {"W": glorot_uniform(k3, self.feature_dim, self.n_classes),
                    "b": jnp.zeros((self.n_classes,), dtype=jnp.float32)},
        }

    def apply(self, params, x):
        h = jax.nn.silu(x @ params["l1"]["W"] + params["l1"]["b"])
        features = jax.nn.silu(h @ params["l2"]["W"] + params["l2"]["b"])
        logits = features @ params["out"]["W"] + params["out"]["b"]
        return logits, features


EVALUATOR = MNISTEvaluatorMLP(
    h1=CONFIG["CLASSIFIER_HIDDEN_1"],
    feature_dim=CONFIG["CLASSIFIER_FEATURE_DIM"],
)


def classifier_loss(params, x, y):
    logits, _ = EVALUATOR.apply(params, x)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(log_probs[jnp.arange(y.shape[0]), y])


@jax.jit
def classifier_step(params, opt_state, x, y):
    loss, grads = jax.value_and_grad(classifier_loss)(params, x, y)
    grads, grad_norm = clip_tree_by_global_norm(grads, 5.0)
    params, opt_state = adam_update(
        params, grads, opt_state, jnp.asarray(CONFIG["CLASSIFIER_LR"], dtype=jnp.float32)
    )
    return params, opt_state, loss, grad_norm


def classifier_accuracy(params, x_np, y_np, batch_size=2_000):
    correct = 0
    total = 0
    for start in range(0, len(x_np), batch_size):
        x = jnp.asarray(x_np[start:start + batch_size])
        logits, _ = EVALUATOR.apply(params, x)
        pred = np.asarray(jnp.argmax(logits, axis=-1))
        y = y_np[start:start + batch_size]
        correct += int(np.sum(pred == y))
        total += len(y)
    return correct / max(total, 1)


def train_evaluator(seed):
    params = EVALUATOR.init(jax.random.PRNGKey(seed))
    opt_state = adam_init(params)
    rng = np.random.default_rng(seed)
    batch_size = int(CONFIG["CLASSIFIER_BATCH_SIZE"])
    start_time = time.time()

    for epoch in range(1, CONFIG["CLASSIFIER_EPOCHS"] + 1):
        perm = rng.permutation(len(train_x_pixels_np))
        losses = []
        for start in range(0, len(perm), batch_size):
            idx = perm[start:start + batch_size]
            if len(idx) < 2:
                continue
            x = jnp.asarray(train_x_pixels_np[idx])
            y = jnp.asarray(train_labels_np[idx], dtype=jnp.int32)
            params, opt_state, loss, _ = classifier_step(params, opt_state, x, y)
            losses.append(float(loss))
        if epoch % CONFIG["CLASSIFIER_PRINT_EVERY"] == 0:
            val_acc = classifier_accuracy(params, val_x_pixels_np, val_labels_np)
            print(
                f"[Evaluator] epoch {epoch}/{CONFIG['CLASSIFIER_EPOCHS']} | "
                f"loss {np.mean(losses):.5f} | val accuracy {val_acc:.4f}"
            )
    return params, time.time() - start_time


print("Training frozen MNIST evaluator...")
EVALUATOR_PARAMS, EVALUATOR_SECONDS = train_evaluator(CONFIG["SEED"] + 60_000)
EVALUATOR_TEST_ACCURACY = classifier_accuracy(
    EVALUATOR_PARAMS, test_x_pixels_np, test_labels_np
)
EVALUATOR_RECON_ACCURACY = classifier_accuracy(
    EVALUATOR_PARAMS, recon_test_np, test_labels_np
)
print("Evaluator training time:", format_seconds(EVALUATOR_SECONDS))
print(f"Evaluator original-test accuracy : {EVALUATOR_TEST_ACCURACY:.4f}")
print(f"Evaluator PCA-reconstruction acc.: {EVALUATOR_RECON_ACCURACY:.4f}")


@partial(jax.jit, static_argnames=("n_samples", "n_steps"))
def sample_reverse_sde(score_params, key, n_samples, n_steps):
    eps_t = float(CONFIG["TIME_EPS"])
    u0 = 0.0
    u1 = 1.0 - eps_t
    du = (u1 - u0) / float(n_steps)
    k_init, k_scan = jax.random.split(key)
    y0 = jax.random.normal(k_init, (n_samples, N))
    step_keys = jax.random.split(k_scan, n_steps)

    def step(y, inp):
        step_idx, noise_key = inp
        u = u0 + step_idx.astype(jnp.float32) * du
        drift = jax.vmap(lambda yy: reverse_drift(score_params, u, yy))(y)
        gamma = reverse_gamma(u)
        noise = jax.random.normal(noise_key, y.shape)
        y_next = y + drift * du + gamma * jnp.sqrt(du) * noise
        return y_next, None

    indices = jnp.arange(n_steps, dtype=jnp.int32)
    y, _ = jax.lax.scan(step, y0, (indices, step_keys))

    # Final Tweedie-style denoising at t=TIME_EPS.
    t_final = jnp.asarray(eps_t, dtype=jnp.float32)
    eps_hat = eps_predict(
        score_params,
        jnp.full((n_samples,), t_final),
        y,
    )
    z0_hat = (y - sigma_t(t_final) * eps_hat) / alpha_t(t_final)
    return z0_hat


def decode_generated_latents(z):
    x = pca_decode_device(z)
    return jnp.clip(x, -1.0, 1.0)


def evaluator_features_and_probs(x_pixels, batch_size=1_000):
    all_features = []
    all_probs = []
    for start in range(0, x_pixels.shape[0], batch_size):
        x = x_pixels[start:start + batch_size]
        logits, features = EVALUATOR.apply(EVALUATOR_PARAMS, x)
        all_features.append(features)
        all_probs.append(jax.nn.softmax(logits, axis=-1))
    return jnp.concatenate(all_features), jnp.concatenate(all_probs)


def covariance_jax(x):
    x = x - jnp.mean(x, axis=0, keepdims=True)
    return (x.T @ x) / jnp.maximum(x.shape[0] - 1, 1)


def psd_sqrt(matrix, eps=1e-6):
    matrix = 0.5 * (matrix + matrix.T)
    vals, vecs = jnp.linalg.eigh(matrix)
    vals = jnp.maximum(vals, eps)
    return (vecs * jnp.sqrt(vals)[None, :]) @ vecs.T


def frechet_distance(features_real, features_fake):
    mu_r = jnp.mean(features_real, axis=0)
    mu_f = jnp.mean(features_fake, axis=0)
    cov_r = covariance_jax(features_real)
    cov_f = covariance_jax(features_fake)
    sqrt_r = psd_sqrt(cov_r)
    middle = sqrt_r @ cov_f @ sqrt_r
    trace_sqrt = jnp.trace(psd_sqrt(middle))
    fd = (
        jnp.sum((mu_r - mu_f) ** 2)
        + jnp.trace(cov_r)
        + jnp.trace(cov_f)
        - 2.0 * trace_sqrt
    )
    return float(jnp.maximum(fd, 0.0))


def sliced_wasserstein(real_z, fake_z, seed, n_projections=None):
    n_projections = int(n_projections or CONFIG["SWD_PROJECTIONS"])
    n = min(real_z.shape[0], fake_z.shape[0])
    real_z = real_z[:n]
    fake_z = fake_z[:n]
    dirs = jax.random.normal(jax.random.PRNGKey(seed), (n_projections, N))
    dirs = dirs / jnp.maximum(jnp.linalg.norm(dirs, axis=1, keepdims=True), 1e-12)
    real_proj = real_z @ dirs.T
    fake_proj = fake_z @ dirs.T
    real_proj = jnp.sort(real_proj, axis=0)
    fake_proj = jnp.sort(fake_proj, axis=0)
    return float(jnp.mean(jnp.sqrt(jnp.mean((real_proj - fake_proj) ** 2, axis=0))))


def nearest_neighbor_distance(fake_z, train_z, fake_count, train_count):
    fake_z = fake_z[:min(fake_count, fake_z.shape[0])]
    train_z = train_z[:min(train_count, train_z.shape[0])]
    values = []
    chunk = 100
    for start in range(0, fake_z.shape[0], chunk):
        f = fake_z[start:start + chunk]
        dist2 = (
            jnp.sum(f * f, axis=1, keepdims=True)
            + jnp.sum(train_z * train_z, axis=1)[None, :]
            - 2.0 * f @ train_z.T
        )
        values.append(jnp.sqrt(jnp.maximum(jnp.min(dist2, axis=1), 0.0)))
    return float(jnp.mean(jnp.concatenate(values)))


def class_distribution_metrics(probs):
    labels = jnp.argmax(probs, axis=1)
    counts = jnp.bincount(labels, length=10).astype(jnp.float32)
    p = counts / jnp.maximum(jnp.sum(counts), 1.0)
    entropy = -jnp.sum(jnp.where(p > 0, p * jnp.log(p), 0.0))
    normalized_entropy = entropy / jnp.log(10.0)
    coverage = jnp.sum(counts > 0)
    confidence = jnp.mean(jnp.max(probs, axis=1))
    return {
        "class_entropy": float(normalized_entropy),
        "class_coverage": int(coverage),
        "mean_classifier_confidence": float(confidence),
        "class_probabilities": np.asarray(p).tolist(),
    }


def compute_model_metrics(name, params, real_pixels_np, real_z, n_samples, seed):
    start = time.time()
    sample_key = jax.random.PRNGKey(seed)
    fake_z = sample_reverse_sde(
        params,
        sample_key,
        n_samples=int(n_samples),
        n_steps=int(CONFIG["SAMPLER_STEPS"]),
    )
    fake_pixels = decode_generated_latents(fake_z)
    real_pixels = jnp.asarray(real_pixels_np[:n_samples])
    real_features, _ = evaluator_features_and_probs(real_pixels)
    fake_features, fake_probs = evaluator_features_and_probs(fake_pixels)

    metrics = {
        "name": name,
        "mnist_fd": frechet_distance(real_features, fake_features),
        "latent_swd": sliced_wasserstein(
            real_z[:n_samples], fake_z, seed + 11
        ),
        "nearest_train_distance": nearest_neighbor_distance(
            fake_z,
            TRAIN_Z,
            CONFIG["NN_GENERATED_SUBSET"],
            CONFIG["NN_TRAIN_SUBSET"],
        ),
        "sampling_seconds": time.time() - start,
    }
    metrics.update(class_distribution_metrics(fake_probs))
    return metrics, fake_z, fake_pixels


# Phase-A validation generation is used to choose m without touching test images.
print("\nPhase-A validation generative evaluation...")
VALIDATION_METRICS = {}
VALIDATION_SAMPLES = {}
for idx, (name, params) in enumerate(MODEL_PARAMS.items()):
    metrics, fake_z, fake_pixels = compute_model_metrics(
        name,
        params,
        val_x_pixels_np,
        VAL_Z,
        n_samples=CONFIG["VALIDATION_SELECTION_SAMPLES"],
        seed=CONFIG["SEED"] + 70_000,  # same prior/sample noise across models
    )
    VALIDATION_METRICS[name] = metrics
    VALIDATION_SAMPLES[name] = (fake_z, fake_pixels)
    print(
        f"{name:21s} | val MNIST-FD {metrics['mnist_fd']:.4f} | "
        f"val SWD {metrics['latent_swd']:.4f} | entropy {metrics['class_entropy']:.3f}"
    )

accepted_gl_names = [
    f"Gaeta-Lie m={m}" for m in CONFIG["GENERATOR_COUNTS"]
    if f"Gaeta-Lie m={m}" in VALIDATION_METRICS
]
if accepted_gl_names:
    SELECTED_GL_NAME = min(
        accepted_gl_names,
        key=lambda n: (
            VALIDATION_METRICS[n]["mnist_fd"],
            VALIDATION_METRICS[n]["latent_swd"],
        ),
    )
    SELECTED_M = int(SELECTED_GL_NAME.split("=")[-1])
else:
    SELECTED_GL_NAME = None
    SELECTED_M = None

print("Validation-selected Gaeta-Lie branch:", SELECTED_GL_NAME)


# Final test evaluation. The Stage-I and continuation controls are always shown;
# all accepted m branches are also shown so the requested three-case sweep remains visible.
print("\nFinal test generative evaluation...")
TEST_METRICS = {}
TEST_SAMPLES = {}
for name, params in MODEL_PARAMS.items():
    metrics, fake_z, fake_pixels = compute_model_metrics(
        name,
        params,
        test_x_pixels_np,
        TEST_Z,
        n_samples=CONFIG["METRIC_SAMPLES"],
        seed=CONFIG["SEED"] + 80_000,
    )
    TEST_METRICS[name] = metrics
    TEST_SAMPLES[name] = (fake_z, fake_pixels)
    print(
        f"{name:21s} | MNIST-FD {metrics['mnist_fd']:.4f} | "
        f"SWD {metrics['latent_swd']:.4f} | entropy {metrics['class_entropy']:.3f} | "
        f"coverage {metrics['class_coverage']}/10 | NN {metrics['nearest_train_distance']:.3f}"
    )


# Paired sample grids: the same initial Gaussian and Brownian increments were used
# because every model was sampled with the same seed.
def show_sample_grid(name, pixels, n_show=None):
    n_show = int(n_show or CONFIG["DISPLAY_SAMPLES"])
    side = int(math.ceil(math.sqrt(n_show)))
    pixels_np = np.asarray(pixels[:n_show])
    fig, axes = plt.subplots(side, side, figsize=(side, side))
    axes = np.asarray(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(pixels_np):
            ax.imshow(pixels_np[i].reshape(28, 28), cmap="gray", vmin=-1, vmax=1)
    plt.suptitle(name)
    plt.tight_layout()
    plt.show()


show_sample_grid("Stage-I paired samples", TEST_SAMPLES["Stage-I"][1])
show_sample_grid("Score-only continuation paired samples", TEST_SAMPLES["Continuation"][1])
if SELECTED_GL_NAME is not None:
    show_sample_grid(
        f"{SELECTED_GL_NAME} paired samples",
        TEST_SAMPLES[SELECTED_GL_NAME][1],
    )


# Top library coefficients for the selected branch.
if SELECTED_M is not None:
    W_selected_np = np.asarray(DISCOVERED[SELECTED_M]["W"])
    mean_abs = np.mean(np.abs(W_selected_np), axis=0)
    top = np.argsort(-mean_abs)[:20]
    print("\nTop 20 basis templates by mean |coefficient| in selected branch")
    for rank, idx in enumerate(top, 1):
        print(
            f"{rank:2d}. mean|W|={mean_abs[idx]:.5f} | "
            f"{BASIS_FAMILIES[idx]:22s} | {BASIS_NAMES[idx]}"
        )


# Optional Phase-B confirmation. This reruns Stage I, selected-m discovery,
# continuation, and Gaeta-Lie refinement for three independent seeds.
def run_confirmation_seed(seed, selected_m, lambda_gl):
    print("\n" + "=" * 88)
    print(f"THREE-SEED CONFIRMATION: seed={seed}")
    print("=" * 88)

    stage1 = train_score_model(seed, verbose=True)
    base_params = stage1["ema_params"]
    branches, stats, discovery_seconds = discover_generator_branches(
        base_params, (selected_m,), seed=seed + 30_000, verbose=True
    )
    disc = branches[selected_m]
    continuation = train_continuation(
        base_params,
        seed=seed + 50_000,
        updates=CONFIG["REFINE_UPDATES"],
        verbose=True,
    )
    if disc["diagnostics"]["accepted"]:
        gl = train_gl_branch(
            base_params,
            disc["W"],
            lambda_gl,
            seed=seed + 50_000,
            updates=CONFIG["REFINE_UPDATES"],
            verbose=True,
        )
        gl_params = gl["ema_params"]
    else:
        print("Adequacy gate abstained in this confirmation seed.")
        gl = None
        gl_params = base_params

    models = {
        "Stage-I": base_params,
        "Continuation": continuation["ema_params"],
        f"Gaeta-Lie m={selected_m}": gl_params,
    }
    out = {
        "seed": int(seed),
        "gate_accepted": bool(disc["diagnostics"]["accepted"]),
        "models": {},
    }
    for name, params in models.items():
        metrics, _, _ = compute_model_metrics(
            name,
            params,
            test_x_pixels_np,
            TEST_Z,
            n_samples=CONFIG["METRIC_SAMPLES"],
            seed=seed + 80_000,
        )
        metrics["test_dsm"] = evaluate_dsm(params, TEST_Z, seed=seed + 90_000)
        out["models"][name] = metrics
        print(
            f"[seed={seed}] {name:21s} | FD {metrics['mnist_fd']:.4f} | "
            f"SWD {metrics['latent_swd']:.4f} | DSM {metrics['test_dsm']:.6f}"
        )
    return out


CONFIRMATION_RESULTS = []
if CONFIG["RUN_MULTI_SEED_CONFIRMATION"]:
    if SELECTED_M is None:
        print("Multi-seed confirmation skipped because no Gaeta-Lie branch passed the gate.")
    else:
        for seed in CONFIG["CONFIRMATION_SEEDS"]:
            CONFIRMATION_RESULTS.append(
                run_confirmation_seed(seed, SELECTED_M, SELECTED_LAMBDA_GL)
            )

        print("\nThree-seed aggregate")
        names = ["Stage-I", "Continuation", f"Gaeta-Lie m={SELECTED_M}"]
        for name in names:
            fd = np.array([r["models"][name]["mnist_fd"] for r in CONFIRMATION_RESULTS])
            swd = np.array([r["models"][name]["latent_swd"] for r in CONFIRMATION_RESULTS])
            dsm = np.array([r["models"][name]["test_dsm"] for r in CONFIRMATION_RESULTS])
            print(
                f"{name:21s} | MNIST-FD {fd.mean():.4f} ± {fd.std(ddof=1):.4f} | "
                f"SWD {swd.mean():.4f} ± {swd.std(ddof=1):.4f} | "
                f"DSM {dsm.mean():.6f} ± {dsm.std(ddof=1):.6f}"
            )


# Final compact report and machine-readable JSON.
FINAL_RESULTS = {
    "config": {
        "fast_mode": bool(CONFIG["FAST_MODE"]),
        "latent_dim": N,
        "generator_counts": list(CONFIG["GENERATOR_COUNTS"]),
        "selected_lambda_gl": float(SELECTED_LAMBDA_GL),
        "selected_m": None if SELECTED_M is None else int(SELECTED_M),
        "metric_samples": int(CONFIG["METRIC_SAMPLES"]),
        "sampler_steps": int(CONFIG["SAMPLER_STEPS"]),
    },
    "representation": {
        "pca_explained_variance": float(PCA_EXPLAINED_RATIO),
        "pca_test_mse": float(pca_test_mse),
        "evaluator_test_accuracy": float(EVALUATOR_TEST_ACCURACY),
        "evaluator_pca_reconstruction_accuracy": float(EVALUATOR_RECON_ACCURACY),
    },
    "adequacy": {
        str(m): DISCOVERED[m]["diagnostics"] for m in CONFIG["GENERATOR_COUNTS"]
    },
    "post_refinement": POST_REFINE_DIAGNOSTICS,
    "branch_determining_residuals": {str(k): v for k, v in BRANCH_DET_DIAGNOSTICS.items()},
    "validation_metrics": VALIDATION_METRICS,
    "test_metrics": TEST_METRICS,
    "confirmation": CONFIRMATION_RESULTS,
}

with open(CONFIG["RESULTS_JSON"], "w") as f:
    json.dump(FINAL_RESULTS, f, indent=2)

print("\n" + "=" * 104)
print("FINAL GAETA-LIE MNIST-64 REPORT")
print("=" * 104)
print(f"PCA-64 explained variance       : {PCA_EXPLAINED_RATIO:.6f}")
print(f"PCA-64 test pixel MSE           : {pca_test_mse:.6e}")
print(f"Evaluator test / recon accuracy : {EVALUATOR_TEST_ACCURACY:.4f} / {EVALUATOR_RECON_ACCURACY:.4f}")
print(f"Shared selected lambda_GL       : {SELECTED_LAMBDA_GL}")
print(f"Validation-selected m           : {SELECTED_M}")
print("\nTest metrics (lower MNIST-FD and SWD are better)")
print("model                 MNIST-FD     latent-SWD   entropy  coverage  confidence  NN-distance")
for name, metrics in TEST_METRICS.items():
    print(
        f"{name:21s} {metrics['mnist_fd']:10.4f} {metrics['latent_swd']:13.4f} "
        f"{metrics['class_entropy']:9.3f} {metrics['class_coverage']:9d}/10 "
        f"{metrics['mean_classifier_confidence']:10.3f} "
        f"{metrics['nearest_train_distance']:11.3f}"
    )

if SELECTED_GL_NAME is not None:
    gl = TEST_METRICS[SELECTED_GL_NAME]
    cont = TEST_METRICS["Continuation"]
    fd_gain = 100.0 * (cont["mnist_fd"] - gl["mnist_fd"]) / max(cont["mnist_fd"], 1e-12)
    swd_gain = 100.0 * (cont["latent_swd"] - gl["latent_swd"]) / max(cont["latent_swd"], 1e-12)
    det_before = BRANCH_DET_DIAGNOSTICS[SELECTED_M]["stage1"]
    det_after = BRANCH_DET_DIAGNOSTICS[SELECTED_M]["gaeta_lie"]
    det_gain = BRANCH_DET_DIAGNOSTICS[SELECTED_M]["reduction_vs_stage1_pct"]
    print("\nMatched-control improvements of selected Gaeta-Lie branch")
    print(f"  MNIST-FD reduction vs continuation : {fd_gain:+.2f}%")
    print(f"  latent-SWD reduction vs continuation: {swd_gain:+.2f}%")
    print(f"  determining-residual reduction      : {det_gain:+.2f}%")

    success = (
        gl["mnist_fd"] < cont["mnist_fd"]
        and gl["latent_swd"] < cont["latent_swd"]
        and det_after < det_before
        and POST_REFINE_DIAGNOSTICS[SELECTED_GL_NAME]["val_dsm"]
            <= 1.05 * POST_REFINE_DIAGNOSTICS["Continuation"]["val_dsm"]
    )
    print("  Operational success gate            :", "PASS" if success else "FAIL")
else:
    print("\nOperational success gate: ABSTAIN — no m branch passed adequacy.")

print("\nResults JSON written to:", CONFIG["RESULTS_JSON"])
print("For a reportable final result, FAST_MODE must be False.")
print("For the planned Phase-B evidence, RUN_MULTI_SEED_CONFIRMATION must be True.")
